# pipe_nuevo — 06 Optuna TRES: solo productos "magicos" (de `residuos_2.ipynb`)

Copia de `06_Optuna.ipynb` (mismo motor: split train/val/test con gap
anti-leakage, los 11 chequeos, WAPE por producto, `_features.json` en vez de
regex, variable respuesta `clase_tn`/`clase_tn_norm`/`clase_tn_delta`, cluster
DTW opcional, early stopping) con UN solo agregado:
`PARAM['archivo_productos_magicos']` filtra `df_sup`/`df_infer` a los
`product_id` que `nat_exp/residuos_2.ipynb` identifico como "magicos" --
aquellos donde el modelo complejo (a nivel producto-mes) le gana al baseline
en validacion. La idea: para esos productos especificamente, ver si bajar a
cliente-producto (shares, edad, recencia, vecinos, peso acumulado, cluster
DTW -- todo lo que tiene pipe_nuevo y que a nivel producto-mes no existe)
mejora todavia mas el WAPE, en vez de gastar ese esfuerzo en productos donde
ya sabemos que un baseline simple alcanza.

El resto del notebook (leakage, Optuna, por-cluster opcional, guardado) es
identico a `06_Optuna.ipynb` -- es el mismo motor, con menos filas.


In [ ]:
import gc, json, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
RUTA_FE  = BUCKET / "datasets_fe"      # entrada: lo que dejan 02_FE/03_Escalado/05_DTW_clusters
RUTA_EXP = BUCKET / "exp_pipe_nuevo"   # salida: una carpeta por experimento
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"\nDatasets de 03_Escalado disponibles en {RUTA_FE.name}/:")
for p in sorted(RUTA_FE.glob("preprocesado_*_pipeNuevo.parquet")):
    print(f"  - {p.name}")
print(f"\nClusters DTW disponibles ({RUTA_FE.name}/):")
for p in sorted(RUTA_FE.glob("clusters_pc_*.parquet")):
    print(f"  - {p.name}")


def rango_meses(desde: int, hasta: int) -> list:
    """Lista de periodos AAAAMM consecutivos, inclusive."""
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


assert desplazar_meses(201912, -2) == 201910
assert desplazar_meses(201901, -1) == 201812


### Palancas


In [ ]:
PARAM = {
    # ── PALANCA 0: dataset de entrada (nombre exacto, de la lista de arriba) ──
    'dataset_fe': None,   # None = el mas reciente en RUTA_FE

    # ── PALANCA 0b: productos magicos (lo nuevo de este notebook) ─────────
    # El productos_magicos.json que escribe nat_exp/residuos_2.ipynb (en la
    # MISMA carpeta datasets_fe). Filtra df_sup/df_infer a esos product_id
    # antes de armar FEATURES o el split -- todo lo de abajo corre igual,
    # solo que con menos filas.
    'archivo_productos_magicos': 'productos_magicos.json',

    # ── PALANCA 1: VARIABLE RESPUESTA ────────────────────────────────────
    # 'clase_tn' | 'clase_tn_norm' | 'clase_tn_delta'
    'target': 'clase_tn_norm',

    # ── PALANCA 2: cluster DTW (opcional) ─────────────────────────────────
    # Nombre exacto de un clusters_pc_*.parquet (de 05_DTW_clusters), o None.
    'archivo_clusters': None,
    # False -> la etiqueta de cluster entra como UNA categorica mas de un
    #          modelo unico (si archivo_clusters esta seteado).
    # True  -> un study de Optuna INDEPENDIENTE por cluster (K veces mas caro:
    #          cada cluster busca sus propios hiperparametros).
    'entrenar_por_cluster': False,

    # ── PALANCA 3: horizonte (DEBE coincidir con el que uso 02_FE) ───────
    'horizonte': 2,

    # ── PALANCA 4: MESES DE TRAIN / VAL / TEST ────────────────────────────
    # Mismo criterio que pipe/03_Optuna.ipynb: gap >= horizonte entre train->val
    # y val->test (lo valida el chequeo de leakage, no se puede saltear).
    'meses_train': rango_meses(201701, 201808),
    'meses_val':   [201810, 201811],
    'meses_test':  [201901],

    'reentrenar_con_val_para_test': True,

    # ── PALANCA 5: muestreo de clientes en TRAIN (None = todos) ──────────
    # Igual criterio que el pipe viejo: hash deterministico por customer_id,
    # conserva la historia completa del cliente elegido. val/test NUNCA se
    # muestrean.
    'muestreo_clientes': None,

    # ── PALANCA 6: WAPE agregado por producto (como evalua la competencia) ──
    'wape_por_producto': True,

    # ── PALANCA 7: objetivo de LightGBM ───────────────────────────────────
    'objective_lgbm': 'regression',
    'tweedie_optimizar': True,

    # ── PALANCA 8: trials nuevos por corrida (se acumulan en el study) ───
    'n_trials': 20,
    'backup_cada_n_trials': 5,

    # ── PALANCA 9: regularizacion ('normal' | 'fuerte') ───────────────────
    'regularizacion': 'normal',

    # ── PALANCA 10: techo de arboles + early stopping ─────────────────────
    # n_estimators YA NO es hiperparametro de Optuna: se entrena hasta este
    # techo, con early stopping contra MESES_VAL, y el mejor trial deja
    # grabado cuantos arboles le alcanzaron de verdad (best_iteration).
    'techo_arboles': 500,
    'early_stopping_rounds': 50,

    # ── PALANCA 11: peso por recencia (None = todos los meses pesan igual) ──
    'decay_recencia': None,

    # ── PALANCA 12: sampling extra de filas de train (None = todas) ──────
    'sampling_frac': None,

    # ── PALANCA 13: features a EXCLUIR a mano ─────────────────────────────
    'features_excluir': [],

    # ── PALANCA 14: sufijo libre para diferenciar corridas ────────────────
    'sufijo': '',

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
    'semilla': 102191,
}

TARGETS_VALIDOS = {'clase_tn': 'nivel', 'clase_tn_norm': 'norm', 'clase_tn_delta': 'delta'}
if PARAM['target'] not in TARGETS_VALIDOS:
    raise ValueError(f"target invalido: {PARAM['target']!r}. Opciones: {list(TARGETS_VALIDOS)}")
TARGET = PARAM['target']
TARGET_KIND = TARGETS_VALIDOS[TARGET]
H = PARAM['horizonte']


### Dataset: se lee el `_features.json` de `03_Escalado` (no un regex de nombre)


In [ ]:
disp = sorted(RUTA_FE.glob("preprocesado_*_pipeNuevo.parquet"))
if not disp:
    raise FileNotFoundError(f"No hay preprocesado_*_pipeNuevo.parquet en {RUTA_FE}. "
                            f"Corre 01_Preprocesamiento -> 02_FE -> 03_Escalado primero.")
nombre_fe = PARAM['dataset_fe'] or max(disp, key=lambda p: p.stat().st_mtime).name
path_in = RUTA_FE / nombre_fe
path_meta = RUTA_FE / nombre_fe.replace(".parquet", "_features.json")
if not path_in.exists():
    raise FileNotFoundError(f"No existe {path_in}.\nDisponibles: {[p.name for p in disp]}")
if not path_meta.exists():
    raise FileNotFoundError(f"No existe {path_meta} (el sidecar que escribe 03_Escalado). "
                            f"Sin el no se sabe con que PARAM se genero el dataset.")

with open(path_meta, encoding="utf-8") as f:
    META_FE = json.load(f)
PARAM_FE = META_FE["param"]
METODO = PARAM_FE["metodo_escalado"]
GRANULARIDAD = PARAM_FE["granularidad"]

print(f"dataset_fe : {nombre_fe}")
print(f"generado con: metodo_escalado={METODO}  granularidad={GRANULARIDAD}  "
      f"salto_delta={PARAM_FE.get('salto_delta')}  horizonte={PARAM_FE['horizonte']}")
if PARAM_FE["horizonte"] != H:
    raise ValueError(f"PARAM['horizonte']={H} no coincide con el horizonte de 02_FE "
                     f"({PARAM_FE['horizonte']}). Tienen que ser el mismo numero.")


### Nombre del experimento


In [ ]:
_v, _t = PARAM['meses_val'], PARAM['meses_test']
TAG_SPLIT = f"val{_v[0]}-{_v[-1]}_test{_t[0]}-{_t[-1]}"

_ex = sorted(PARAM['features_excluir'])
TAG_EXCL = ("__excl-" + "-".join(_ex)) if 0 < len(_ex) <= 3 else (f"__excl{len(_ex)}feats" if _ex else "")
TAG_CLI = f"__cli1de{PARAM['muestreo_clientes']}" if PARAM.get('muestreo_clientes') else ""
_arb = int(PARAM['techo_arboles'])
TAG_ARB = f"__arb{_arb}" if _arb != 500 else ""
TAG_CLUSTER = "__porCluster" if (PARAM['archivo_clusters'] and PARAM['entrenar_por_cluster']) else (
    "__clusterFeat" if PARAM['archivo_clusters'] else "")
TAG_MAGICOS = "__soloMagicos" if PARAM['archivo_productos_magicos'] else ""

EXPERIMENTO_BASE = (
    f"{nombre_fe.replace('.parquet', '')}"
    f"__y-{TARGET_KIND}"
    f"__{PARAM['objective_lgbm']}"
    f"__{TAG_SPLIT}"
    + TAG_EXCL + TAG_CLI + TAG_ARB + TAG_CLUSTER + TAG_MAGICOS
    + (f"__{PARAM['sufijo']}" if PARAM['sufijo'] else "")
)

print(f"EXPERIMENTO_BASE: {EXPERIMENTO_BASE}")


### Carga: Float32 + muestreo de clientes, todo en la pasada lazy

Mismo criterio de memoria que `pipe/03_Optuna.ipynb`: Float32 en las features
(LightGBM las discretiza en 255 bins igual), muestreo de clientes por hash
determinístico SOLO en train, `scan_parquet` + `collect` ya filtrado.


In [ ]:
t0 = time.time()

CTX_F64 = {'B0', 'B1', 'tn0_norm', 'tn0', 'clase_tn', 'clase_tn_norm', 'clase_tn_delta'}

lf = pl.scan_parquet(path_in)
_schema = lf.collect_schema()
COLS_ALL = list(_schema.keys())

if TARGET not in COLS_ALL:
    raise ValueError(f"El dataset no tiene la columna {TARGET!r}. Columnas clase_* "
                     f"disponibles: {[c for c in COLS_ALL if c.startswith('clase_')]}")

_f64 = [c for c, t in _schema.items() if t == pl.Float64]
_a_f32 = [c for c in _f64 if c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
print(f"Dataset: {len(COLS_ALL)} columnas")
print(f"Periodos: {periodos[0]} -> {periodos[-1]}  ({len(periodos)} meses)")
print(f"Float64 -> Float32: {len(_a_f32)} de {len(_f64)} columnas")

MESES_INFER = periodos[-H:]

_N_CLI = PARAM.get('muestreo_clientes')
_es_eval = pl.col('periodo').is_in(sorted(set(PARAM['meses_val']) | set(PARAM['meses_test'])))
_es_train = pl.col('periodo').is_in(sorted(set(PARAM['meses_train'])))

if _N_CLI and 'customer_id' in COLS_ALL:
    _cli_ok = pl.col('customer_id').hash(seed=PARAM['semilla']) % int(_N_CLI) == 0
    _filtro_sup = pl.col(TARGET).is_not_null() & (_es_eval | (_es_train & _cli_ok))
else:
    _filtro_sup = pl.col(TARGET).is_not_null()

df_infer = lf.filter(pl.col(TARGET).is_null() & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup = lf.filter(_filtro_sup).collect()
n_infer = df_infer.height

_total = lf.select(pl.len()).collect().item()
_sup_disponibles = lf.select(pl.col(TARGET).is_not_null().sum()).collect().item()

print(f"\nSupervisadas en el parquet : {_sup_disponibles:,}")
print(f"Supervisadas CARGADAS      : {df_sup.height:,}  ({100*df_sup.height/_sup_disponibles:.0f}%)"
      + (f"   [1 de cada {_N_CLI} clientes en train]" if _N_CLI else "   [sin muestreo]"))
print(f"Inferencia (target nulo)   : {n_infer:,}   -> periodos {sorted(df_infer['periodo'].unique().to_list())}")
print(f"Descartadas (nulo por fin de vida): {_total - _sup_disponibles - n_infer:,}")
print(f"\nRAM del split: {(df_sup.estimated_size() + df_infer.estimated_size()) / 1e9:.2f} GB")
print(f"[{time.time()-t0:.0f}s]")


### Filtro a productos "magicos" (lo nuevo de este notebook)

`nat_exp/residuos_2.ipynb` ya decidio, por WAPE en validacion A NIVEL
PRODUCTO-MES, cuales productos se benefician de un modelo complejo. Se filtra
ACA, antes de armar `FEATURES` o el split -- todo lo de abajo (leakage,
Optuna, por-cluster) corre exactamente igual, solo que sobre este
subconjunto.


In [ ]:
if PARAM['archivo_productos_magicos']:
    path_mag = RUTA_FE / PARAM['archivo_productos_magicos']
    if not path_mag.exists():
        raise FileNotFoundError(f"No existe {path_mag}. Corre nat_exp/residuos_2.ipynb primero, "
                                f"o poné PARAM['archivo_productos_magicos'] = None.")
    with open(path_mag, encoding="utf-8") as f:
        META_MAGICOS = json.load(f)
    PRODUCTOS_MAGICOS = set(META_MAGICOS["product_ids"])
    if not PRODUCTOS_MAGICOS:
        raise ValueError(f"{path_mag.name} no tiene ningun producto magico "
                         f"(ver el WAPE de {META_MAGICOS.get('esquema')} vs "
                         f"{META_MAGICOS.get('baseline')} en residuos_2). Nada que filtrar aca.")

    antes_sup, antes_inf = df_sup.height, df_infer.height
    df_sup = df_sup.filter(pl.col("product_id").is_in(PRODUCTOS_MAGICOS))
    df_infer = df_infer.filter(pl.col("product_id").is_in(PRODUCTOS_MAGICOS))

    print(f"productos magicos: {len(PRODUCTOS_MAGICOS)} "
         f"(de {META_MAGICOS.get('n_total', '?')} totales en residuos_2, "
         f"esquema={META_MAGICOS.get('esquema')} vs baseline={META_MAGICOS.get('baseline')})")
    print(f"filas supervisadas : {antes_sup:,} -> {df_sup.height:,}")
    print(f"filas de inferencia: {antes_inf:,} -> {df_infer.height:,}")
    if df_sup.height == 0 or df_infer.height == 0:
        raise RuntimeError("El filtro de productos magicos dejo 0 filas -- revisa que "
                           "dataset_fe y archivo_productos_magicos sean del mismo universo "
                           "de productos (solo_productos_target, etc).")
else:
    print("PARAM['archivo_productos_magicos'] = None -> no se filtra (igual que 06_Optuna.ipynb)")


### Cluster DTW (opcional)

Join por `(product_id, customer_id)`. Los pares sin cluster (no clusterizados,
o `granularidad='p'`) quedan en el centinela `-1` -- nunca se pierden filas
por esto.


In [ ]:
COL_CLUSTER = None          # nombre de trabajo, siempre "cluster" si se usa
COL_CLUSTER_ORIG = None     # nombre real en el parquet de 05_DTW_clusters (cluster_pc_k{K});
                            # es el que hay que guardar en resultado.json para poder releer
                            # el archivo de clusters mas adelante (07_Entrenamiento_final).
if PARAM['archivo_clusters']:
    path_cl = RUTA_FE / PARAM['archivo_clusters']
    if not path_cl.exists():
        disp_cl = sorted(RUTA_FE.glob("clusters_pc_*.parquet"))
        raise FileNotFoundError(f"No existe {path_cl}.\nDisponibles: {[p.name for p in disp_cl]}")
    clu = pl.read_parquet(path_cl)
    COL_CLUSTER_ORIG = next((c for c in clu.columns if c.startswith("cluster_pc_k")), None)
    if COL_CLUSTER_ORIG is None:
        raise ValueError(f"{path_cl.name} no tiene ninguna columna cluster_pc_k*. "
                         f"Columnas: {clu.columns}")
    clu = clu.select(["product_id", "customer_id", COL_CLUSTER_ORIG]).rename({COL_CLUSTER_ORIG: "cluster"})
    COL_CLUSTER = "cluster"

    antes_sup, antes_inf = df_sup.height, df_infer.height
    df_sup = (df_sup.join(clu, on=["product_id", "customer_id"], how="left")
                    .with_columns(pl.col("cluster").fill_null(-1)))
    df_infer = (df_infer.join(clu, on=["product_id", "customer_id"], how="left")
                        .with_columns(pl.col("cluster").fill_null(-1)))
    assert df_sup.height == antes_sup and df_infer.height == antes_inf, \
        "el join de clusters duplico filas -- el parquet de clusters tiene pares repetidos"

    _sin_cluster = int((df_sup["cluster"] == -1).sum())
    print(f"cluster leido de: {path_cl.name}  (columna {COL_CLUSTER})")
    print(f"pares sin cluster (centinela -1): {_sin_cluster:,} de {df_sup.height:,} filas de train/val/test")
    print(df_sup.group_by("cluster").agg(pl.len().alias("n_filas")).sort("cluster"))
else:
    print("PARAM['archivo_clusters'] = None -> no se usa informacion de cluster")


### Features: que entra y que no

Prohibido: identificadores, el eje temporal, y TODAS las columnas `clase_*`
(las tres son el mismo dato en otra escala -- usar cualquiera que no sea el
target elegido seria leakage puro). En modo `entrenar_por_cluster`, la columna
`cluster` TAMBIEN queda prohibida (es constante dentro de cada subconjunto).


In [ ]:
COLS_ID = [c for c in ['product_id', 'customer_id', 'periodo', 'm', 'periodo_objetivo']
          if c in COLS_ALL]
COLS_CLASE = [c for c in df_sup.columns if c.startswith('clase_')]
COLS_PROHIBIDAS = set(COLS_ID) | set(COLS_CLASE)
if COL_CLUSTER and PARAM['entrenar_por_cluster']:
    COLS_PROHIBIDAS.add(COL_CLUSTER)

COLS_DISPONIBLES = list(df_sup.columns)
FEATURES = [c for c in COLS_DISPONIBLES
           if c not in COLS_PROHIBIDAS and c not in PARAM['features_excluir']]

TIPOS_TEXTO = (pl.Utf8, pl.String, pl.Categorical, pl.Enum, pl.Boolean)
_cols_cat_pedidas = list(PARAM['cols_categoricas'])
if COL_CLUSTER and not PARAM['entrenar_por_cluster']:
    _cols_cat_pedidas.append(COL_CLUSTER)
CAT_AUTO = [c for c in FEATURES
           if df_sup.schema[c] in TIPOS_TEXTO and c not in _cols_cat_pedidas]
CAT_FEATURES = [c for c in _cols_cat_pedidas if c in FEATURES] + CAT_AUTO

print(f"Features ({len(FEATURES)})")
print(f"Categoricas: {CAT_FEATURES}")
print(f"\nExcluidas por prohibidas ({len(COLS_PROHIBIDAS)}): {sorted(COLS_PROHIBIDAS)}")
if PARAM['features_excluir']:
    print(f"Excluidas a mano: {PARAM['features_excluir']}")


### Meses de train/val/test y control de leakage


In [ ]:
periodos_sup = sorted(df_sup['periodo'].unique().to_list())
set_sup = set(periodos_sup)

MESES_TRAIN = sorted(set(PARAM['meses_train']) & set_sup)
MESES_VAL   = sorted(set(PARAM['meses_val'])   & set_sup)
MESES_TEST  = sorted(set(PARAM['meses_test'])  & set_sup)

for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(f"{nombre} quedo vacio. Periodos supervisados disponibles: "
                         f"{periodos_sup[0]}..{periodos_sup[-1]}. Revisa PARAM['meses_{nombre.lower()}'].")
    print(f"{nombre:6s} ({len(ms):2d} meses): {ms[0]} .. {ms[-1]}")


In [ ]:
leak = {'errores': [], 'warnings': [], 'ok': []}


def _err(msg):
    leak['errores'].append(msg)
    print(f"  [ERROR]   {msg}")


def _ok(msg):
    leak['ok'].append(msg)
    print(f"  [ok]      {msg}")


print("CONTROL DE DATA LEAKAGE")
print("=" * 72)

intrusas = sorted(set(FEATURES) & COLS_PROHIBIDAS)
if intrusas:
    _err(f"columnas prohibidas dentro de FEATURES: {intrusas}")
else:
    _ok(f"ninguna de las {len(COLS_PROHIBIDAS)} columnas prohibidas esta en FEATURES")

clase_en_x = sorted(set(FEATURES) & set(COLS_CLASE))
if clase_en_x:
    _err(f"columnas clase_* usadas como feature: {clase_en_x}")
else:
    _ok(f"ninguna clase_* es feature; {TARGET} se usa solo como target")

for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, 'train', 'val'),
                     (MESES_VAL, MESES_TEST, 'val', 'test')):
    gap = a_indice_mes(min(b)) - a_indice_mes(max(a))
    if gap < H:
        _err(f"gap {na}->{nb} = {gap} mes(es) < horizonte {H}")
    else:
        _ok(f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es) >= horizonte {H}")

if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_indice_mes(min(MESES_TEST)) - a_indice_mes(max(MESES_TRAIN + MESES_VAL))
    if gap_tv < H:
        _err(f"reentrenar_con_val_para_test=True pero el gap (train+val)->test es {gap_tv} < {H}")
    else:
        _ok(f"gap (train+val) -> test = {gap_tv} >= {H}")

for (na, a), (nb, b) in ((('train', MESES_TRAIN), ('val', MESES_VAL)),
                         (('train', MESES_TRAIN), ('test', MESES_TEST)),
                         (('val', MESES_VAL),     ('test', MESES_TEST))):
    inter = sorted(set(a) & set(b))
    if inter:
        _err(f"{na} y {nb} comparten los meses {inter}")
    else:
        _ok(f"{na} y {nb} son disjuntos")

if max(MESES_TRAIN) >= min(MESES_VAL):
    _err(f"train llega a {max(MESES_TRAIN)}, igual o posterior al inicio de val {min(MESES_VAL)}")
if max(MESES_VAL) >= min(MESES_TEST):
    _err(f"val llega a {max(MESES_VAL)}, igual o posterior al inicio de test {min(MESES_TEST)}")
if max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST):
    _ok("orden cronologico correcto: train < val < test")

_N_CORR = 500_000
_mc = df_sup if df_sup.height <= _N_CORR else df_sup.sample(n=_N_CORR, seed=PARAM['semilla'])
y_chk = _mc[TARGET].to_numpy().astype(np.float64)
sospechosas = []
num_feats = [c for c in FEATURES if _mc.schema[c] in
            (pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
             pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64)]
for c in num_feats:
    x = _mc[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(y_chk)
    if ok.sum() < 100:
        continue
    xs, ys = x[ok], y_chk[ok]
    if xs.std() == 0 or ys.std() == 0:
        continue
    r = float(np.corrcoef(xs, ys)[0, 1])
    if abs(r) > 0.999:
        sospechosas.append((c, round(r, 6)))
if sospechosas:
    _err(f"features con |corr| > 0.999 contra el target: {sospechosas}")
else:
    _ok(f"ninguna de las {len(num_feats)} features numericas correlaciona >0.999 con el target")
del _mc, y_chk

if df_sup.filter(pl.col(TARGET).is_null()).height:
    _err("quedaron filas con target nulo en el set supervisado")
else:
    _ok(f"{n_infer:,} filas de inferencia (target nulo) apartadas en df_infer")

periodos_infer = set(df_infer['periodo'].unique().to_list())
solapa = sorted(periodos_infer & (set(MESES_TRAIN) | set(MESES_VAL) | set(MESES_TEST)))
if solapa:
    _err(f"periodos de inferencia usados en train/val/test: {solapa}")
else:
    _ok("los periodos de inferencia no se usan para entrenar, validar ni testear")

print("=" * 72)
leak['resumen'] = f"{len(leak['errores'])} error(es)"
print(leak['resumen'])
if leak['errores']:
    raise RuntimeError(f"Control de leakage FALLIDO: {leak['errores']}")
print("Control superado.")


### Reconstruccion a toneladas y WAPE

`reconstruir_nivel()` es mas simple que la del pipe viejo: TODOS los metodos
de escalado de pipe_nuevo (`mediana`/`media`/`zscore`/`rolling_mean`) usan la
misma forma afin `norm = (valor - B0) / B1` -- no existe el caso especial
`'recta'` (ajuste lineal) del pipe viejo.


In [ ]:
def reconstruir_nivel(pred, df_ctx) -> np.ndarray:
    """Pasa la prediccion del modelo a toneladas, segun la variable respuesta elegida.

    df_ctx (pandas) debe traer B0, B1 y (si target='clase_tn_delta') tn0_norm,
    alineadas por fila con `pred`.
    """
    pred = np.asarray(pred, dtype=np.float64)
    if TARGET_KIND == 'nivel':
        return pred
    if TARGET_KIND == 'delta':
        # clase_tn_delta = clase_tn_norm - tn0_norm -> volvemos a la escala normalizada
        pred = pred + df_ctx['tn0_norm'].to_numpy(dtype=np.float64)
    B0 = df_ctx['B0'].to_numpy(dtype=np.float64)
    B1 = df_ctx['B1'].to_numpy(dtype=np.float64)
    B1_safe = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1_safe + B0


def wape(y_real_tn, y_pred_tn, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Con por_producto=True agrega por product_id primero
    (asi lo evalua la competencia)."""
    y_real = np.asarray(y_real_tn, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred_tn, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        ids = np.asarray(product_ids)
        orden, inv = np.unique(ids, return_inverse=True)
        y_real = np.bincount(inv, weights=y_real, minlength=len(orden))
        y_pred = np.bincount(inv, weights=y_pred, minlength=len(orden))
    den = np.abs(y_real).sum()
    return float('nan') if den == 0 else float(np.abs(y_real - y_pred).sum() / den)


# Chequeo de sanidad: reconstruir el TARGET REAL tiene que devolver clase_tn.
_m = df_sup.head(min(20_000, df_sup.height)).to_pandas()
_rec = reconstruir_nivel(_m[TARGET].to_numpy(), _m)
_err_max = float(np.nanmax(np.abs(_rec - _m['clase_tn'].to_numpy())))
print(f"Round-trip de reconstruccion (target -> toneladas): error maximo = {_err_max:.10f}")
if _err_max > 1e-6:
    raise RuntimeError(f"La reconstruccion a toneladas no cierra (error {_err_max}). "
                       f"Revisa que METODO={METODO!r} sea el que uso 03_Escalado.")
print("Reconstruccion validada: el WAPE se mide en toneladas reales.")
del _m


### A pandas una sola vez

Las columnas de contexto (`B0`/`B1`/`tn0_norm`/`clase_tn` + ids) viajan con
cada fila para poder reconstruir toneladas y medir WAPE sin volver a tocar
polars.


In [ ]:
if PARAM['objective_lgbm'] in ('tweedie', 'poisson') and TARGET_KIND != 'nivel':
    raise ValueError(f"objective_lgbm={PARAM['objective_lgbm']!r} exige target >= 0, "
                     f"pero target={TARGET!r} puede tener valores negativos. Usa target='clase_tn'.")

IDS = [c for c in ['product_id', 'customer_id'] if c in COLS_ALL]
COLS_CTX = [c for c in ['B0', 'B1', 'tn0_norm', 'clase_tn'] + IDS
           + ([COL_CLUSTER] if COL_CLUSTER else []) if c in COLS_ALL or c == COL_CLUSTER]

_cols_pd = sorted(set(FEATURES + COLS_CTX + [TARGET, 'periodo']))
_cats = [c for c in CAT_FEATURES if c in _cols_pd]

df_pd = (df_sup.select(_cols_pd)
              .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats])
              .to_pandas())
df_infer_pd = (df_infer.select([c for c in _cols_pd if c in df_infer.columns])
                       .with_columns([pl.col(c).cast(pl.Categorical)
                                      for c in _cats if c in df_infer.columns])
                       .to_pandas())
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')
    if c in df_infer_pd.columns:
        df_infer_pd[c] = df_infer_pd[c].astype('category').cat.set_categories(df_pd[c].cat.categories)

print(f"df_pd (supervisado): {df_pd.shape}   |   df_infer_pd: {df_infer_pd.shape}")

for _v in ('df_sup', 'df_infer'):
    globals().pop(_v, None)
gc.collect()
print(f"polars liberado; df_pd en RAM: {df_pd.memory_usage(deep=True).sum() / 1e9:.2f} GB")


### El experimento, empaquetado en una funcion

`correr_experimento()` hace TODO lo que en el pipe viejo estaba suelto en el
notebook: busqueda de Optuna con early stopping, reentreno final con la
cantidad de arboles que encontro el mejor trial, evaluacion val/test,
baseline naive, guardado de predicciones + `resultado.json` + importancia +
graficos de Optuna. Se llama UNA vez (modo pooled) o UNA vez POR CLUSTER (modo
`entrenar_por_cluster`) -- misma funcion, distinto subconjunto de filas.


In [ ]:
import lightgbm as lgb
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def espacio_hiper(trial, techo_arboles, regularizacion, objective_lgbm,
                  tweedie_optimizar, semilla):
    base = {
        'objective': objective_lgbm, 'metric': 'mae', 'verbosity': -1,
        'boosting_type': 'gbdt', 'seed': semilla, 'subsample_freq': 1, 'n_jobs': -1,
        'n_estimators': int(techo_arboles),   # techo: early stopping decide cuantos usa
    }
    if regularizacion == 'fuerte':
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 8, 64),
            'max_depth':         trial.suggest_int('max_depth', 3, 7),
            'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
            'max_depth':         trial.suggest_int('max_depth', 3, 12),
            'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    if objective_lgbm == 'tweedie' and tweedie_optimizar:
        base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return base


def pesos_recencia(periodos_serie, decay):
    if decay is None:
        return None
    ps = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(ps)}
    n = len(ps)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values


def entrenar(df_pd_sub, params, meses_tr, features, cat_features, target,
            decay_recencia=None, sampling_frac=None, semilla=0,
            eval_meses=None, early_stopping_rounds=50):
    """Entrena con las filas cuyo periodo esta en meses_tr. Si eval_meses no es
    None, entrena con early stopping contra esos meses (params['n_estimators']
    actua como TECHO, no como cantidad fija)."""
    idx = df_pd_sub.index[df_pd_sub['periodo'].isin(meses_tr)]
    if len(idx) == 0:
        raise ValueError(f'Sin filas de entrenamiento para los meses {meses_tr}')
    if sampling_frac is not None and sampling_frac < 1.0:
        rng = np.random.default_rng(semilla)
        idx = idx[rng.choice(len(idx), size=int(len(idx) * sampling_frac), replace=False)]

    X = df_pd_sub.loc[idx, features]
    y = df_pd_sub.loc[idx, target]
    w = pesos_recencia(df_pd_sub.loc[idx, 'periodo'], decay_recencia)

    modelo = lgb.LGBMRegressor(**params)
    fit_kwargs = dict(sample_weight=w, categorical_feature=cat_features)
    if eval_meses:
        idx_ev = df_pd_sub.index[df_pd_sub['periodo'].isin(eval_meses)]
        if len(idx_ev) == 0:
            raise ValueError(f'Sin filas de evaluacion para early stopping en {eval_meses}')
        X_ev = df_pd_sub.loc[idx_ev, features]
        y_ev = df_pd_sub.loc[idx_ev, target]
        fit_kwargs['eval_set'] = [(X_ev, y_ev)]
        fit_kwargs['callbacks'] = [lgb.early_stopping(early_stopping_rounds, verbose=False)]
    modelo.fit(X, y, **fit_kwargs)
    del X, y, w
    gc.collect()
    return modelo


def predecir(modelo, df_eval, features, ids, cols_ctx, target, horizonte):
    pred = modelo.predict(df_eval[features])
    ctx = df_eval[[c for c in cols_ctx if c in df_eval.columns]].reset_index(drop=True)
    pred_tn = np.maximum(reconstruir_nivel(pred, ctx), 0.0)

    out = df_eval[ids + ['periodo']].copy()
    out['periodo_objetivo'] = out['periodo'].map(lambda p: desplazar_meses(p, horizonte))
    out['y_pred_target'] = pred
    out['tn_pred'] = pred_tn
    if 'clase_tn' in df_eval.columns:
        out['tn_real'] = df_eval['clase_tn'].values
        out['y_real_target'] = df_eval[target].values
    return out


def evaluar(modelo, df_pd_sub, meses_ev, features, ids, cols_ctx, target, horizonte,
           wape_por_producto, por_mes=True):
    df_ev = df_pd_sub[df_pd_sub['periodo'].isin(meses_ev)]
    if len(df_ev) == 0:
        return float('nan'), {}, None
    pred = predecir(modelo, df_ev, features, ids, cols_ctx, target, horizonte)
    por_mes_d = {}
    if por_mes:
        for m in sorted(meses_ev):
            sub = pred[pred['periodo'] == m]
            if len(sub):
                por_mes_d[int(m)] = wape(sub['tn_real'], sub['tn_pred'], sub['product_id'],
                                         wape_por_producto)
    glob = wape(pred['tn_real'], pred['tn_pred'], pred['product_id'], wape_por_producto)
    return glob, por_mes_d, pred


print("funciones de entrenamiento listas")


In [ ]:
def correr_experimento(df_pd_sub, df_infer_pd_sub, features, cat_features,
                       experimento, dir_out, cluster_val=None):
    """Un study de Optuna completo (busqueda + reentreno val/test + guardado)
    sobre el subconjunto de filas que se le pase. Se llama una vez en modo
    pooled, o una vez por cluster en modo entrenar_por_cluster."""
    dir_out.mkdir(parents=True, exist_ok=True)
    db_local = Path.home() / f"optuna_{experimento}.db"
    db_bucket = RUTA_EXP / "optuna_db" / f"{experimento}.db"
    db_bucket.parent.mkdir(parents=True, exist_ok=True)
    if db_bucket.exists() and not db_local.exists():
        shutil.copy(db_bucket, db_local)
        print(f"  Study recuperado del bucket: {db_bucket}")
    storage = f"sqlite:///{db_local}"

    def objective(trial):
        params = espacio_hiper(trial, PARAM['techo_arboles'], PARAM['regularizacion'],
                               PARAM['objective_lgbm'], PARAM['tweedie_optimizar'], PARAM['semilla'])
        modelo = entrenar(df_pd_sub, params, MESES_TRAIN, features, cat_features, TARGET,
                          decay_recencia=PARAM['decay_recencia'], sampling_frac=PARAM['sampling_frac'],
                          semilla=PARAM['semilla'], eval_meses=MESES_VAL,
                          early_stopping_rounds=PARAM['early_stopping_rounds'])
        score, por_mes, _ = evaluar(modelo, df_pd_sub, MESES_VAL, features, IDS, COLS_CTX,
                                    TARGET, H, PARAM['wape_por_producto'])
        if np.isnan(score):
            raise optuna.TrialPruned()
        bi = getattr(modelo, 'best_iteration_', None)
        trial.set_user_attr('best_iteration', int(bi) if bi else int(params['n_estimators']))
        for m, v in por_mes.items():
            trial.set_user_attr(f'wape_val_{m}', v)
        return float(score)

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                study_name=experimento, storage=storage, load_if_exists=True)
    print(f"  experimento   : {experimento}")
    print(f"  trials previos: {len(study.trials)}  ->  corriendo {PARAM['n_trials']} nuevos")

    def respaldar():
        try:
            tmp = db_bucket.with_suffix('.db.tmp')
            shutil.copy(db_local, tmp)
            tmp.replace(db_bucket)
            return True
        except Exception as e:
            print(f"    [aviso] no se pudo respaldar: {e}")
            return False

    cada = max(1, int(PARAM['backup_cada_n_trials']))
    with tqdm(total=PARAM['n_trials'], desc=experimento[:40]) as pbar:
        def cb(st, tr):
            pbar.update(1)
            try:
                pbar.set_postfix({'mejor WAPE': f"{st.best_value:.5f}"})
            except ValueError:
                pass
            if pbar.n % cada == 0:
                respaldar()
        try:
            study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[cb])
        except KeyboardInterrupt:
            print("  interrumpido a mano: respaldando lo hecho hasta ahora...")
        finally:
            if respaldar():
                print(f"  study respaldado en {db_bucket}")

    print(f"  mejor WAPE (val): {study.best_value:.5f}")
    mejores_params = espacio_hiper(optuna.trial.FixedTrial(study.best_params), PARAM['techo_arboles'],
                                   PARAM['regularizacion'], PARAM['objective_lgbm'],
                                   PARAM['tweedie_optimizar'], PARAM['semilla'])
    n_arboles_final = int(study.best_trial.user_attrs.get('best_iteration')
                          or PARAM['techo_arboles'])
    mejores_params_fijos = dict(mejores_params)
    mejores_params_fijos['n_estimators'] = n_arboles_final
    print(f"  arboles del mejor trial (early stopping): {n_arboles_final}")

    modelo_val = entrenar(df_pd_sub, mejores_params_fijos, MESES_TRAIN, features, cat_features,
                          TARGET, decay_recencia=PARAM['decay_recencia'], semilla=PARAM['semilla'])
    wape_val, wape_val_mes, pred_val = evaluar(modelo_val, df_pd_sub, MESES_VAL, features, IDS,
                                               COLS_CTX, TARGET, H, PARAM['wape_por_producto'])
    print(f"  VAL   wape={wape_val:.5f}")

    meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
    modelo_final = entrenar(df_pd_sub, mejores_params_fijos, meses_fit_test, features, cat_features,
                            TARGET, decay_recencia=PARAM['decay_recencia'], semilla=PARAM['semilla'])
    wape_test, wape_test_mes, pred_test = evaluar(modelo_final, df_pd_sub, MESES_TEST, features, IDS,
                                                  COLS_CTX, TARGET, H, PARAM['wape_por_producto'])
    print(f"  TEST  wape={wape_test:.5f}")

    def wape_naive_en(meses):
        _n = df_pd_sub[df_pd_sub['periodo'].isin(meses)]
        if len(_n) == 0 or 'tn0' not in _n.columns:
            return float('nan')
        return wape(_n['clase_tn'].to_numpy(), _n['tn0'].to_numpy(),
                   _n['product_id'].to_numpy(), PARAM['wape_por_producto'])

    naive_val, naive_test = wape_naive_en(MESES_VAL), wape_naive_en(MESES_TEST)
    print(f"  naive wape: val={naive_val:.5f}  test={naive_test:.5f}")

    pl.from_pandas(pred_val).write_parquet(dir_out / 'predicciones_val.parquet')
    pl.from_pandas(pred_test).write_parquet(dir_out / 'predicciones_test.parquet')

    if len(df_infer_pd_sub) == 0:
        print("  sin filas de inferencia para este subconjunto.")
    else:
        pred_infer = predecir(modelo_final, df_infer_pd_sub, features, IDS, COLS_CTX, TARGET, H)
        pl.from_pandas(pred_infer).write_parquet(dir_out / 'predicciones_inferencia.parquet')
        print(f"  predicciones de inferencia: {len(pred_infer):,} filas")

    imp = pd.DataFrame({
        'feature': modelo_final.feature_name_,
        'gain':    modelo_final.booster_.feature_importance(importance_type='gain'),
        'split':   modelo_final.booster_.feature_importance(importance_type='split'),
    }).sort_values('gain', ascending=False).reset_index(drop=True)
    _suma_gain = imp['gain'].sum()
    imp['gain_pct'] = 100 * imp['gain'] / _suma_gain if _suma_gain > 0 else 0.0
    imp.to_csv(dir_out / 'importancia.csv', index=False)
    sin_uso = imp.loc[imp['gain'] == 0, 'feature'].tolist()
    print(f"  features con gain > 0: {int((imp['gain'] > 0).sum())} de {len(imp)}")

    study.trials_dataframe().to_csv(dir_out / 'trials.csv', index=False)

    resultado = {
        'experimento': experimento, 'cluster': cluster_val,
        'dataset_fe': nombre_fe, 'param_fe': PARAM_FE,
        'target': TARGET, 'target_kind': TARGET_KIND, 'metodo_normalizacion': METODO,
        'horizonte': H,
        'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
        'reentrenar_con_val_para_test': PARAM['reentrenar_con_val_para_test'],
        'metrica': 'wape_toneladas' + ('_por_producto' if PARAM['wape_por_producto'] else '_por_fila'),
        'wape_mejor_trial': study.best_value,
        'wape_val': wape_val, 'wape_val_por_mes': wape_val_mes,
        'wape_test': wape_test, 'wape_test_por_mes': wape_test_mes,
        'wape_naive_val': naive_val, 'wape_naive_test': naive_test,
        'n_trials_total': len(study.trials),
        'n_estimators_final': n_arboles_final,
        'objective_lgbm': PARAM['objective_lgbm'], 'regularizacion': PARAM['regularizacion'],
        'techo_arboles': PARAM['techo_arboles'],
        'early_stopping_rounds': PARAM['early_stopping_rounds'],
        'muestreo_clientes': PARAM.get('muestreo_clientes'),
        'n_filas': int(len(df_pd_sub)),
        'n_filas_train': int(df_pd_sub['periodo'].isin(MESES_TRAIN).sum()),
        'n_features': len(features), 'features': features, 'cat_features': cat_features,
        'archivo_clusters': PARAM['archivo_clusters'], 'col_cluster': COL_CLUSTER_ORIG,
        'archivo_productos_magicos': PARAM['archivo_productos_magicos'],
        'hiperparametros': study.best_params, 'semilla': PARAM['semilla'],
        'leakage': leak['resumen'], 'features_sin_uso': sin_uso,
    }
    with open(dir_out / 'resultado.json', 'w', encoding='utf-8') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
    shutil.copy(db_local, db_bucket)
    print(f"  guardado en {dir_out}")
    return resultado


### Corrida: pooled o por-cluster segun `PARAM['entrenar_por_cluster']`


In [ ]:
t0 = time.time()

if COL_CLUSTER and PARAM['entrenar_por_cluster']:
    valores_cluster = sorted(df_pd[COL_CLUSTER].unique().tolist())
    print(f"Modo POR CLUSTER: {len(valores_cluster)} clusters -> {valores_cluster}")
    DIR_BASE = RUTA_EXP / f"{EXPERIMENTO_BASE}"
    DIR_BASE.mkdir(parents=True, exist_ok=True)

    resultados_cluster = []
    for cl in valores_cluster:
        sub = df_pd[df_pd[COL_CLUSTER] == cl].reset_index(drop=True)
        sub_infer = (df_infer_pd[df_infer_pd[COL_CLUSTER] == cl].reset_index(drop=True)
                    if COL_CLUSTER in df_infer_pd.columns else df_infer_pd.iloc[0:0])
        exp_cl = f"{EXPERIMENTO_BASE}__cluster{cl}"
        dir_cl = DIR_BASE / f"cluster{cl}"
        print(f"\n{'='*70}\ncluster {cl}: {len(sub):,} filas de train/val/test, "
              f"{len(sub_infer):,} de inferencia\n{'='*70}")
        r = correr_experimento(sub, sub_infer, FEATURES, CAT_FEATURES, exp_cl, dir_cl, cluster_val=int(cl))
        resultados_cluster.append(r)

    manifest = {
        'experimento_base': EXPERIMENTO_BASE,
        'archivo_clusters': PARAM['archivo_clusters'], 'col_cluster': COL_CLUSTER_ORIG,
        'clusters': [int(c) for c in valores_cluster],
        'resultados': [str(DIR_BASE / f"cluster{c}" / 'resultado.json') for c in valores_cluster],
        'wape_test_ponderado': float(
            sum(r['wape_test'] * r['n_filas'] for r in resultados_cluster if r['wape_test'] == r['wape_test'])
            / sum(r['n_filas'] for r in resultados_cluster)
        ) if resultados_cluster else None,
    }
    with open(DIR_BASE / 'manifest.json', 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)
    print(f"\nmanifest: {DIR_BASE / 'manifest.json'}")
    print(f"WAPE test ponderado por tamano de cluster: {manifest['wape_test_ponderado']}")
else:
    DIR_OUT = RUTA_EXP / EXPERIMENTO_BASE
    print(f"Modo POOLED (un solo modelo)")
    resultado = correr_experimento(df_pd, df_infer_pd, FEATURES, CAT_FEATURES,
                                   EXPERIMENTO_BASE, DIR_OUT)

print(f"\n[{time.time()-t0:.0f}s]")


### Leaderboard: todos los experimentos corridos hasta ahora


In [ ]:
filas = []
for f_res in sorted(RUTA_EXP.glob('**/resultado.json')):
    r = json.load(open(f_res, encoding='utf-8'))
    _nt = r.get('wape_naive_test')
    filas.append({
        'experimento':      r['experimento'],
        'cluster':          r.get('cluster'),
        'wape_test':        r.get('wape_test'),
        'wape_val':         r.get('wape_val'),
        'brecha_test_val':  (round(r['wape_test'] - r['wape_val'], 5)
                             if r.get('wape_test') is not None and r.get('wape_val') is not None
                             else None),
        'mejora_vs_naive_%': (round(100 * (_nt - r['wape_test']) / _nt, 2)
                              if _nt and r.get('wape_test') is not None else None),
        'target':           r['target_kind'],
        'objective':        r['objective_lgbm'],
        'regularizacion':   r['regularizacion'],
        'n_estimators_final': r.get('n_estimators_final'),
        'n_filas':          r.get('n_filas'),
        'n_features':       r['n_features'],
        'n_trials':         r['n_trials_total'],
        'leakage':          r['leakage'],
        'archivo':          str(f_res.relative_to(RUTA_EXP)),
    })

leaderboard = pd.DataFrame(filas).sort_values('wape_test').reset_index(drop=True)
leaderboard.to_csv(RUTA_EXP / 'leaderboard.csv', index=False)
print(f"{len(leaderboard)} resultado(s) en {RUTA_EXP/'leaderboard.csv'}")
leaderboard
